# Multimodal Fusion Tutorial:

#Task 1: Image + Text Classification on UPMC Food-101


We use the **UPMC Food-101** dataset: ~90,000 food photos scraped from the web, each paired
with the noisy text (title/description) found next to it on the page it came from, labeled
with one of 101 food categories. Because the text is noisy and the images are often visually
ambiguous (many dishes look alike), neither modality is reliable on its own.

**Roadmap:**

1. **Setup & data** — download the dataset, inspect it, get it into a usable form.
2. **Image-only classification.** Fine-tune a CNN (ResNet-18) on the photos alone.
   Trained with **cross-entropy loss**.
3. **Text-only classification.** Fine-tune a small transformer (DistilBERT) on the
   text alone. Also trained with **cross-entropy loss**.
4. **Multimodal fusion.** Combine both modalities in three different ways and compare:
   - **Early / feature-level fusion** — concatenate the two modalities' feature vectors and train a joint classifier end-to-end, and evaluate its performance.

   - **Late / decision-level fusion** — keep the Module 2 and Module 3 models separate, and combine their predictions after the fact using a weighted average and evaluate their performance.

   - **Attention-based fusion (bidirectional cross-attention)** —  let the two modalities attend to each other before the final classification, instead of just concatenating them.

5. **Comparison** — put all five numbers side by side and discuss what the pattern tells us.


> **Before you start:** in Colab, go to `Runtime → Change runtime type` and select a **GPU**.
> Fine-tuning a CNN and a transformer on CPU will be painfully slow.


## Module 1: Setup & Data Download

In [ ]:
# torch/torchvision ship pre-installed on Colab GPU runtimes. We only need to add
# the Hugging Face + Kaggle libraries on top.
!pip -q install -U transformers kagglehub


In [ ]:
import os
import glob
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms

from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type == "cpu":
    print("WARNING: no GPU detected — Runtime > Change runtime type > GPU is strongly recommended.")


## Download and inspect the dataset

We'll pull the pre-processed UPMC Food-101 mirror from Kaggle via `kagglehub`. The first time
you run this in a session it will ask you to authenticate with a Kaggle account (this opens a
login flow — you need a free Kaggle account, no special access request required).


In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download("gianmarco96/upmcfood101")
print("Dataset downloaded to:", dataset_path)


**Exercise for you**

Comment the above code and write your own code to download the dataset

This pre-processed mirror is generally organized as:

```
images/train/<class_name>/<file>.jpg
images/test/<class_name>/<file>.jpg
texts/train_titles.csv   # columns: image_path, text, food
texts/test_titles.csv
```

Different uploads of this dataset occasionally store `image_path` a little differently (just a
filename vs. a path that already includes the class folder), so instead of hard-coding one
assumption, the next cells auto-detect the layout and verify it actually resolves to real
files** before we build anything on top of it.

In [ ]:
def find_first(patterns):
    for p in patterns:
        matches = glob.glob(os.path.join(dataset_path, p), recursive=True)
        if matches:
            return matches[0]
    return None

images_dir = find_first(["images", "**/images"])
train_csv = find_first(["texts/train_titles.csv", "**/train_titles.csv"])
test_csv = find_first(["texts/test_titles.csv", "**/test_titles.csv"])

print("images_dir:", images_dir)
print("train_csv :", train_csv)
print("test_csv  :", test_csv)

assert images_dir and train_csv and test_csv, (
    "Could not auto-detect the dataset layout. Print `os.listdir(dataset_path)` "
    "and set images_dir / train_csv / test_csv by hand."
)


In [ ]:
COLUMNS = ["image_path", "text", "food"]
train_df = pd.read_csv(train_csv, names=COLUMNS, header=None)
test_df = pd.read_csv(test_csv, names=COLUMNS, header=None)

print("train:", train_df.shape, " test:", test_df.shape)
train_df.head()


In [ ]:
def resolve_image_path(images_dir, split, image_path, food):
    """
    Try the plausible ways `image_path` might relate to a real file on disk and
    return whichever one actually exists. This makes the notebook robust to the
    small layout differences between mirrors of this dataset instead of silently
    assuming one and failing later with a confusing FileNotFoundError.
    """
    candidates = [
        image_path,
        os.path.join(dataset_path, image_path),
        os.path.join(images_dir, split, image_path),
        os.path.join(images_dir, split, food, image_path),
        os.path.join(images_dir, split, food, os.path.basename(image_path)),
    ]
    for c in candidates:
        if os.path.isfile(c):
            return c
    return None


train_df["full_path"] = train_df.apply(
    lambda r: resolve_image_path(images_dir, "train", r["image_path"], r["food"]), axis=1
)
test_df["full_path"] = test_df.apply(
    lambda r: resolve_image_path(images_dir, "test", r["image_path"], r["food"]), axis=1
)

missing_train = train_df["full_path"].isna().sum()
missing_test = test_df["full_path"].isna().sum()
print(f"Unresolved train images: {missing_train} / {len(train_df)}")
print(f"Unresolved test images:  {missing_test} / {len(test_df)}")

train_df = train_df.dropna(subset=["full_path"]).reset_index(drop=True)
test_df = test_df.dropna(subset=["full_path"]).reset_index(drop=True)

assert len(train_df) > 0 and len(test_df) > 0, "No images resolved — inspect the layout above."


## A quick look at the data

In [ ]:
#Uncomment the code below to visualize the number of samples per class


# print(f"Number of classes: {train_df['food'].nunique()}")

# class_counts = train_df["food"].value_counts()
# plt.figure(figsize=(12, 4))
# class_counts.plot(kind="bar")
# plt.title("Training samples per class (UPMC Food-101)")
# plt.xticks([])
# plt.ylabel("count")
# plt.show()

# print("Text length (characters) stats:")
# print(train_df["text"].str.len().describe())


In [ ]:
# A few example (image, text, label) triples
fig, axes = plt.subplots(2, 3, figsize=(13, 8))
sample = train_df.sample(6, random_state=SEED).reset_index(drop=True)
for ax, (_, row) in zip(axes.flat, sample.iterrows()):
    img = Image.open(row["full_path"]).convert("RGB")
    ax.imshow(img)
    ax.set_title(row["food"], fontsize=10)
    caption = row["text"] if len(row["text"]) <= 60 else row["text"][:57] + "..."
    ax.set_xlabel(caption, fontsize=8)
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()


## Data Configuration

The full training split has ~68,000 images across 101 classes — fine-tuning a CNN and a
transformer and two fusion models on all of it would take hours. For a tutorial we subsample
a fixed number of examples per class so everything below runs in a few minutes on a Colab GPU.
Set `samples_per_class_train` / `samples_per_class_test` to `None` to use the full dataset once
you're ready to train for real.


In [ ]:
CONFIG = {
    "samples_per_class_train": 40,   # set to None to use the full training split
    "samples_per_class_test": 10,    # set to None to use the full test split
    "img_size": 224,
    "max_text_len": 32,
    "batch_size": 32,
    "epochs": 3,
    "lr_image": 1e-4,     # CNN classifier head + fine-tuning
    "lr_text": 2e-5,      # transformer fine-tuning (transformers need a much smaller LR)
    "lr_fusion": 2e-5,    # joint models fine-tune a transformer too, so use the same small LR
    "text_model_name": "distilbert-base-uncased",
}


def subsample_per_class(df, n, seed=SEED):
    if n is None:
        return df
    parts = [g.sample(min(len(g), n), random_state=seed) for _, g in df.groupby("food")]
    return pd.concat(parts).reset_index(drop=True)


train_sub = subsample_per_class(train_df, CONFIG["samples_per_class_train"])
test_sub = subsample_per_class(test_df, CONFIG["samples_per_class_test"])

classes = sorted(train_df["food"].unique())
label2id = {c: i for i, c in enumerate(classes)}
id2label = {i: c for c, i in label2id.items()}
NUM_CLASSES = len(classes)
print("Number of classes:", NUM_CLASSES)

train_sub["label"] = train_sub["food"].map(label2id)
test_sub["label"] = test_sub["food"].map(label2id)

train_final, val_final = train_test_split(
    train_sub, test_size=0.15, random_state=SEED, stratify=train_sub["label"]
)
train_final = train_final.reset_index(drop=True)
val_final = val_final.reset_index(drop=True)
test_final = test_sub.reset_index(drop=True)

print(f"train={len(train_final)}  val={len(val_final)}  test={len(test_final)}")


##Exercise for you

- Change the learning rate of image and evaluate the performance
- Change the learning rate of image and evaluate the performance
- Change the learning rate of image and evaluate the performance

## Shared dataset and training utilities

We define **one** `Dataset` class that returns the image tensor, the tokenized text, and the
label for every example.


In [ ]:
image_transform_train = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
image_transform_eval = transforms.Compose([
    transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

tokenizer = AutoTokenizer.from_pretrained(CONFIG["text_model_name"])


class FoodDataset(Dataset):
    def __init__(self, df, tokenizer, image_transform, max_len):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.image_transform = image_transform
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image = Image.open(row["full_path"]).convert("RGB")
        image = self.image_transform(image)

        enc = self.tokenizer(
            str(row["text"]),
            padding="max_length",
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt",
        )

        return {
            "image": image,
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": int(row["label"]),
        }


pin = device.type == "cuda"
train_ds = FoodDataset(train_final, tokenizer, image_transform_train, CONFIG["max_text_len"])
val_ds = FoodDataset(val_final, tokenizer, image_transform_eval, CONFIG["max_text_len"])
test_ds = FoodDataset(test_final, tokenizer, image_transform_eval, CONFIG["max_text_len"])

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=2, pin_memory=pin)
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=pin)
test_loader = DataLoader(test_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2, pin_memory=pin)


##Exercise for you

- Increase the number of epochs and evaluate the performance

In [ ]:
# `results` collects the final test-set numbers for every model so Module 5 can compare them.
results = {}


def run_epoch(model, loader, optimizer, forward_fn, train=True):
    model.train() if train else model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for batch in loader:
            labels = batch["label"].to(device)
            logits = forward_fn(model, batch)
            loss = F.cross_entropy(logits, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * labels.size(0)
            all_preds.append(logits.argmax(dim=1).detach().cpu())
            all_labels.append(labels.cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average="macro")
    return total_loss / len(loader.dataset), acc, f1


def fit(model, train_loader, val_loader, forward_fn, epochs, lr, name):
    model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    for epoch in range(epochs):
        train_loss, _, _ = run_epoch(model, train_loader, optimizer, forward_fn, train=True)
        val_loss, val_acc, val_f1 = run_epoch(model, val_loader, optimizer, forward_fn, train=False)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        print(f"[{name}] epoch {epoch + 1}/{epochs} - train_loss {train_loss:.3f} "
              f"- val_loss {val_loss:.3f} - val_acc {val_acc:.3f}")
    return history


def evaluate_and_record(model, loader, forward_fn, name):
    _, acc, f1 = run_epoch(model, loader, optimizer=None, forward_fn=forward_fn, train=False)
    results[name] = {"test_acc": acc, "test_f1": f1}
    print(f"[{name}] TEST accuracy={acc:.3f}  macro-F1={f1:.3f}")
    return acc, f1


### The loss function used for training

**Cross-Entropy**

All end-to-end trained classification models in the tutorial use the same loss: `torch.nn.functional.cross_entropy(logits, labels)`. This makes the comparison focus on the architecture and fusion mechanism rather than changing the objective.
Cross-entropy encourages the model to assign high probability to the correct food class and penalizes confident incorrect predictions strongly.
more harshly than unconfident wrong answers (because of the `log`).

## Module 2: Image-only classification

This baseline uses a pretrained ResNet-18. ResNet-18 is an 18-layer deep convolutional neural network used mainly for image classification and computer vision tasks.
Its original ImageNet classification layer is replaced with a new linear layer containing 101 outputs.



In [ ]:
image_model = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
image_model.fc = nn.Linear(image_model.fc.in_features, NUM_CLASSES)


def image_forward(model, batch):
    images = batch["image"].to(device)
    return model(images)


history_img = fit(image_model, train_loader, val_loader, image_forward,
                   CONFIG["epochs"], CONFIG["lr_image"], name="image-only")
evaluate_and_record(image_model, test_loader, image_forward, "Image-only (ResNet-18)")


### Look at some actual predictions before moving on





In [ ]:
@torch.no_grad()
def collect_predictions(model, loader, df, forward_fn, id2label):
    """Run `model` over `loader` and return one row per example with its true/predicted
    label, the full probability vector, and enough info to look the example back up (`df` must
    be in the same order as `loader`, i.e. built with shuffle=False)."""
    model.eval()
    rows = []
    offset = 0
    for batch in loader:
        logits = forward_fn(model, batch)
        probs = F.softmax(logits, dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)
        labels = batch["label"].numpy()
        for i in range(len(labels)):
            row = df.iloc[offset + i]
            rows.append({
                "full_path": row["full_path"],
                "text": row["text"],
                "true_id": int(labels[i]),
                "pred_id": int(preds[i]),
                "true_label": id2label[labels[i]],
                "pred_label": id2label[preds[i]],
                "true_prob": float(probs[i, labels[i]]),
                "pred_prob": float(probs[i, preds[i]]),
                "probs": probs[i],
                "correct": bool(preds[i] == labels[i]),
            })
        offset += len(labels)
    return pd.DataFrame(rows)


image_preds_df = collect_predictions(image_model, test_loader, test_final, image_forward, id2label)
print(f"Test accuracy check: {image_preds_df['correct'].mean():.3f}  "
      f"(should match the [Image-only (ResNet-18)] number printed above)")


In [ ]:
def show_prediction_example(example, ax_img, ax_bar, title, top_k=3):
    img = Image.open(example["full_path"]).convert("RGB")
    ax_img.imshow(img)
    ax_img.axis("off")
    verdict = "CORRECT" if example["correct"] else "WRONG"
    ax_img.set_title(
        f"{title}  [{verdict}]\ntrue: {example['true_label']}\npred: {example['pred_label']}",
        fontsize=10,
    )

    probs = example["probs"]
    top_idx = np.argsort(probs)[::-1][:top_k]
    top_labels = [id2label[i] for i in top_idx]
    top_vals = probs[top_idx]
    # highlight the bar for the true class in green so it's easy to spot how close it came
    colors = ["#2ca02c" if i == example["true_id"] else "#9e9e9e" for i in top_idx]

    y_pos = range(len(top_labels))[::-1]
    ax_bar.barh(y_pos, top_vals, color=colors)
    ax_bar.set_yticks(y_pos)
    ax_bar.set_yticklabels(top_labels, fontsize=9)
    ax_bar.set_xlim(0, 1)
    ax_bar.set_xlabel("predicted probability")
    ax_bar.set_title(f"top-{top_k} class probabilities", fontsize=9)


correct_pool = image_preds_df[image_preds_df["correct"]]
wrong_pool = image_preds_df[~image_preds_df["correct"]].copy()

assert len(correct_pool) > 0, "The model got nothing right — check training before continuing."

# A clean, confident correct example.
correct_example = correct_pool.sort_values("pred_prob", ascending=False).iloc[0]

if len(wrong_pool) == 0:
    print("The image-only model made zero mistakes on this (small, subsampled) test set — "
          "try a smaller `samples_per_class_train` or fewer epochs to see a confused example, "
          "or just take this as an unusually easy split.")
    confused_example = None
else:
    # The "confused" example: a WRONG prediction where the true class's own probability came
    # closest to the winning prediction's probability -- i.e. a genuine, close-call mix-up
    # rather than a wild, low-confidence guess.
    wrong_pool["confusion_gap"] = wrong_pool["pred_prob"] - wrong_pool["true_prob"]
    confused_example = wrong_pool.sort_values("confusion_gap").iloc[0]

if confused_example is not None:
    fig, axes = plt.subplots(2, 2, figsize=(11, 9), gridspec_kw={"width_ratios": [1, 1.1]})
    show_prediction_example(correct_example, axes[0, 0], axes[0, 1], "A confident correct prediction")
    show_prediction_example(confused_example, axes[1, 0], axes[1, 1], "A confused prediction")
    plt.tight_layout()
    plt.show()

    print(f"Scraped text for the confused example (unused by this image-only model): "
          f"\"{confused_example['text']}\"")
    print(f"True class '{confused_example['true_label']}' got probability "
          f"{confused_example['true_prob']:.2f}; the model instead predicted "
          f"'{confused_example['pred_label']}' with probability {confused_example['pred_prob']:.2f}.")
    print("Why might the model make this mistake? Most likely because these two categories are "
          "visually similar enough that the photo alone doesn't contain enough information to "
          "tell them apart -- which is exactly the gap a second modality (the text) can fill.")


## Module 3: Text-only classification

This baseline uses DistilBERT transformer and considers only text input. BERT is Bidirectional Encoder Representations from Transformers and  DistilBERT is a smaller, faster, and lighter version of the BERT language model, developed by Hugging Face.
DistilBERT is loaded with a 101-class sequence-classification head.



In [ ]:
text_model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["text_model_name"], num_labels=NUM_CLASSES
)


def text_forward(model, batch):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    return model(input_ids=input_ids, attention_mask=attention_mask).logits


history_txt = fit(text_model, train_loader, val_loader, text_forward,
                   CONFIG["epochs"], CONFIG["lr_text"], name="text-only")
evaluate_and_record(text_model, test_loader, text_forward, "Text-only (DistilBERT)")


### Complementary modalities


Suppose the image model predicts the wrong class, but the text model predicts the correct one.
Then the text must contain information the image model failed to capture — some word that
disambiguates what the photo alone couldn't.

Now consider the opposite situation: the caption is short or ambiguous, but the photo is visually
distinctive. Then the image provides information the text does not.

This is what we call **complementary modalities**.


In [ ]:
text_preds_df = collect_predictions(text_model, test_loader, test_final, text_forward, id2label)

# Merge the two models' per-example predictions. Safe as a straight positional join because
# both `test_loader`s were built with shuffle=False over the same `test_final` ordering.
combined = image_preds_df[["full_path", "text", "true_label", "pred_label", "pred_prob", "correct"]].copy()
combined = combined.rename(columns={
    "pred_label": "img_pred", "pred_prob": "img_pred_prob", "correct": "img_correct",
})
combined["img_probs"] = image_preds_df["probs"].values
combined["txt_pred"] = text_preds_df["pred_label"].values
combined["txt_pred_prob"] = text_preds_df["pred_prob"].values
combined["txt_correct"] = text_preds_df["correct"].values
combined["txt_probs"] = text_preds_df["probs"].values

both_correct = int((combined["img_correct"] & combined["txt_correct"]).sum())
img_only = int((combined["img_correct"] & ~combined["txt_correct"]).sum())
txt_only = int((~combined["img_correct"] & combined["txt_correct"]).sum())
both_wrong = int((~combined["img_correct"] & ~combined["txt_correct"]).sum())
total = len(combined)

print(f"Both models correct:            {both_correct:4d}  ({both_correct/total:.1%})")
print(f"Image right, text wrong:        {img_only:4d}  ({img_only/total:.1%})")
print(f"Text right, image wrong:        {txt_only:4d}  ({txt_only/total:.1%})")
print(f"Both models wrong:               {both_wrong:4d}  ({both_wrong/total:.1%})")

recoverable = img_only + txt_only
print(f"\n{recoverable} examples ({recoverable/total:.1%} of the test set) were solved by exactly "
      f"ONE of the two modalities alone. Those are the cases a good fusion model should be able "
      f"to recover, since between the two single-modality models the right answer already exists "
      f"somewhere -- fusion just has to learn when to trust which one.")


In [ ]:
def bar_panel(ax, probs, model_name, pred_label, true_label, top_k=3):
    top_idx = np.argsort(probs)[::-1][:top_k]
    labels = [id2label[i] for i in top_idx]
    vals = probs[top_idx]
    colors = ["#2ca02c" if lbl == true_label else "#9e9e9e" for lbl in labels]
    y = range(len(labels))[::-1]
    ax.barh(y, vals, color=colors)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=8)
    ax.set_xlim(0, 1)
    verdict = "correct" if pred_label == true_label else "WRONG"
    ax.set_title(f"{model_name}: '{pred_label}' ({verdict})", fontsize=8)


def show_complementary_gallery(examples_df, title, max_examples=3):
    """Plot up to `max_examples` rows, each showing one example's photo alongside both
    models' top-3 predicted classes, so complementarity is something you can SEE rather than
    a single summary statistic."""
    n = min(len(examples_df), max_examples)
    if n == 0:
        print(f"{title}: no matching examples in this test subsample -- try a larger "
              f"samples_per_class_test to make one more likely to show up.")
        return

    fig, axes = plt.subplots(n, 3, figsize=(13, 4.0 * n), gridspec_kw={"width_ratios": [1, 1.1, 1.1]})
    axes = np.atleast_2d(axes)  # keep 2D indexing even when n == 1

    for row, (_, example) in enumerate(examples_df.head(n).iterrows()):
        img = Image.open(example["full_path"]).convert("RGB")
        axes[row, 0].imshow(img)
        axes[row, 0].axis("off")
        caption = example["text"] if len(example["text"]) <= 45 else example["text"][:42] + "..."
        axes[row, 0].set_title(f"true: {example['true_label']}\ncaption: \"{caption}\"", fontsize=8)

        bar_panel(axes[row, 1], example["img_probs"], "image model", example["img_pred"], example["true_label"])
        bar_panel(axes[row, 2], example["txt_probs"], "text model", example["txt_pred"], example["true_label"])

    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()


case1_pool = combined[(~combined["img_correct"]) & (combined["txt_correct"])]
case2_pool = combined[(combined["img_correct"]) & (~combined["txt_correct"])]

print(f"'Image wrong, text right' examples available: {len(case1_pool)}")
print(f"'Text wrong, image right' examples available: {len(case2_pool)}")

# Most convincing examples first: for case 1, sort by how confident the (correct) text model
# was; for case 2, by how confident the (correct) image model was.
case1_top = case1_pool.sort_values("txt_pred_prob", ascending=False)
case2_top = case2_pool.sort_values("img_pred_prob", ascending=False)

show_complementary_gallery(
    case1_top, "Case 1 -- image WRONG, text RIGHT: evidence the text provides useful information", max_examples=3
)


## Module 4: Multimodal fusion

Fusion refers to the point at which the image and text branches stop being processed independently and begin to influence the final prediction. The notebook implements feature-level fusion (early), decision-level fusion (late), and attention-based fusion.



  









In [ ]:
class ImageBackbone(nn.Module):
    """ResNet-18 with the final classification layer removed, so forward() returns a 512-d
    feature vector per image instead of class logits."""

    def __init__(self):
        super().__init__()
        resnet = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
        resnet.fc = nn.Identity()
        self.backbone = resnet
        self.out_dim = 512

    def forward(self, images):
        return self.backbone(images)


class TextBackbone(nn.Module):
    """DistilBERT with no classification head; forward() returns the pooled [CLS]-style
    768-d embedding of the input text."""

    def __init__(self, model_name):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.out_dim = self.backbone.config.hidden_size

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        return out.last_hidden_state[:, 0, :]


### Module 4.1— Early / feature-level fusion

 Run each modality through its own backbone to
get one feature vector per modality, concatenate the two vectors, and feed the result into a
joint classifier — training the whole thing (both backbones + classifier) end-to-end with one
loss. Concretely here: `torch.cat([img_feat (512-d), txt_feat (768-d)], dim=1)` → a 1280-d
vector → a small MLP → 101 logits.

In [ ]:
class EarlyFusionModel(nn.Module):
    def __init__(self, num_classes, text_model_name, hidden=512, dropout=0.3):
        super().__init__()
        self.image_backbone = ImageBackbone()
        self.text_backbone = TextBackbone(text_model_name)
        fused_dim = self.image_backbone.out_dim + self.text_backbone.out_dim
        self.classifier = nn.Sequential(
            nn.Linear(fused_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, num_classes),
        )

    def forward(self, images, input_ids, attention_mask):
        img_feat = self.image_backbone(images)
        txt_feat = self.text_backbone(input_ids, attention_mask)
        fused = torch.cat([img_feat, txt_feat], dim=1)
        return self.classifier(fused)


def early_fusion_forward(model, batch):
    images = batch["image"].to(device)
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    return model(images, input_ids, attention_mask)


early_fusion_model = EarlyFusionModel(NUM_CLASSES, CONFIG["text_model_name"])
history_early = fit(early_fusion_model, train_loader, val_loader, early_fusion_forward,
                     CONFIG["epochs"], CONFIG["lr_fusion"], name="early-fusion")
evaluate_and_record(early_fusion_model, test_loader, early_fusion_forward, "Early/feature fusion")


### Module 4.2 — Late / decision-level fusion


Train each modality's classifier completely
separately and only combine them after each has
already produced its own class probabilities. There's no shared backbone and no joint loss.

**A weighted average of the two probability distributions** — `p = α·p_img + (1-α)·p_txt`.
   `α` is not learned by gradient descent, it's chosen by a
  small grid search that directly maximizes validation accuracy.







In [ ]:
@torch.no_grad()
def get_probs(model, loader, forward_fn):
    model.eval()
    all_probs, all_labels = [], []
    for batch in loader:
        logits = forward_fn(model, batch)
        all_probs.append(F.softmax(logits, dim=1).cpu())
        all_labels.append(batch["label"])
    return torch.cat(all_probs).numpy(), torch.cat(all_labels).numpy()


val_img_probs, val_labels = get_probs(image_model, val_loader, image_forward)
val_txt_probs, _ = get_probs(text_model, val_loader, text_forward)
test_img_probs, test_labels = get_probs(image_model, test_loader, image_forward)
test_txt_probs, _ = get_probs(text_model, test_loader, text_forward)

# (a) weighted average of the two probability distributions, weight tuned on val
best_alpha, best_val_acc = 0.5, 0.0
for alpha in np.linspace(0, 1, 21):
    blend = alpha * val_img_probs + (1 - alpha) * val_txt_probs
    acc = accuracy_score(val_labels, blend.argmax(axis=1))
    if acc > best_val_acc:
        best_val_acc, best_alpha = acc, alpha
print(f"Best image weight found on val set: alpha={best_alpha:.2f} (val_acc={best_val_acc:.3f})")

test_blend = best_alpha * test_img_probs + (1 - best_alpha) * test_txt_probs
blend_preds = test_blend.argmax(axis=1)
acc = accuracy_score(test_labels, blend_preds)
f1 = f1_score(test_labels, blend_preds, average="macro")
results["Late fusion (weighted avg)"] = {"test_acc": acc, "test_f1": f1}
print(f"[Late fusion - weighted avg] TEST accuracy={acc:.3f}  macro-F1={f1:.3f}")


### Module 4.3— Attention-based fusion: bidirectional cross-attention

A middle ground that keeps modality-specific
processing and lets the modalities directly influence each other before the final decision, by
having each one **attend to** the other — rather than just being concatenated. We implement this
as genuine **bidirectional cross-attention**: image tokens query the text sequence, text tokens
query the image sequence, and the two resulting context vectors are concatenated for the final
classifier.



With real sequences on both sides, we run cross-attention **in both directions**:

1. **Image → text**: image-region tokens act as queries, while text tokens act as keys and values. Each image region can therefore retrieve information from the text that is relevant to that region.

2. **Text → image**: symmetrically, the text tokens act as queries and the image patches act as
   keys and values. It will address , "which regions of the photo are most relevant to this word?"





In [ ]:
class ImagePatchBackbone(nn.Module):
    """ResNet-18 truncated *before* global average pooling, so forward() returns a sequence
    of spatial feature tokens (one per 32x32 image region) instead of one pooled vector."""

    def __init__(self):
        super().__init__()
        resnet = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
        # keep every layer except the final avgpool + fc -> output is (B, 512, H, W)
        self.features = nn.Sequential(*list(resnet.children())[:-2])
        self.out_dim = 512

    def forward(self, images):
        feat_map = self.features(images)               # (B, 512, H, W), e.g. (B, 512, 7, 7)
        b, c, h, w = feat_map.shape
        tokens = feat_map.flatten(2).transpose(1, 2)    # (B, H*W, C) = (B, 49, 512) patch tokens
        return tokens


class TextTokenBackbone(nn.Module):
    """DistilBERT with no pooling; forward() returns the full per-token sequence instead of
    just the [CLS]-style first token."""

    def __init__(self, model_name):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.out_dim = self.backbone.config.hidden_size

    def forward(self, input_ids, attention_mask):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        return out.last_hidden_state    # (B, L, 768) - one vector per word-piece token


class BiCrossAttentionFusion(nn.Module):
    """Bidirectional cross-attention fusion: image patch tokens attend over the text
    sequence, text tokens attend over the image patches, and the two pooled, attended
    representations are concatenated before the final classifier."""

    def __init__(self, num_classes, text_model_name, proj_dim=256, num_heads=4, dropout=0.3):
        super().__init__()
        self.image_backbone = ImagePatchBackbone()
        self.text_backbone = TextTokenBackbone(text_model_name)
        self.image_proj = nn.Linear(self.image_backbone.out_dim, proj_dim)
        self.text_proj = nn.Linear(self.text_backbone.out_dim, proj_dim)

        # two SEPARATE attention modules: one per direction of cross-attention
        self.img2txt_attn = nn.MultiheadAttention(proj_dim, num_heads, dropout=dropout, batch_first=True)
        self.txt2img_attn = nn.MultiheadAttention(proj_dim, num_heads, dropout=dropout, batch_first=True)

        self.norm_img_attn = nn.LayerNorm(proj_dim)
        self.norm_txt_attn = nn.LayerNorm(proj_dim)

        # a small per-modality feed-forward block after attention, transformer-encoder style
        ffn_dim = proj_dim * 4
        self.img_ffn = nn.Sequential(nn.Linear(proj_dim, ffn_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(ffn_dim, proj_dim))
        self.txt_ffn = nn.Sequential(nn.Linear(proj_dim, ffn_dim), nn.GELU(), nn.Dropout(dropout), nn.Linear(ffn_dim, proj_dim))
        self.norm_img_ffn = nn.LayerNorm(proj_dim)
        self.norm_txt_ffn = nn.LayerNorm(proj_dim)

        self.classifier = nn.Sequential(
            nn.Linear(proj_dim * 2, proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(proj_dim, num_classes),
        )

    def forward(self, images, input_ids, attention_mask):
        img_tokens = self.image_proj(self.image_backbone(images))               # (B, 49, D)
        txt_tokens = self.text_proj(self.text_backbone(input_ids, attention_mask))  # (B, L, D)

        # True where a text position is PADDING, so attention ignores it as a key/value.
        text_key_padding_mask = attention_mask == 0                              # (B, L)

        # --- direction 1: image patches query the text sequence ---
        img_ctx, _ = self.img2txt_attn(
            query=img_tokens, key=txt_tokens, value=txt_tokens,
            key_padding_mask=text_key_padding_mask,
        )
        img_ctx = self.norm_img_attn(img_ctx + img_tokens)                       # residual
        img_ctx = self.norm_img_ffn(img_ctx + self.img_ffn(img_ctx))            # feed-forward + residual
        img_vec = img_ctx.mean(dim=1)                                            # pool 49 tokens -> (B, D)

        # --- direction 2: text tokens query the image patches (all 49 patches are always valid) ---
        txt_ctx, _ = self.txt2img_attn(query=txt_tokens, key=img_tokens, value=img_tokens)
        txt_ctx = self.norm_txt_attn(txt_ctx + txt_tokens)                       # residual
        txt_ctx = self.norm_txt_ffn(txt_ctx + self.txt_ffn(txt_ctx))            # feed-forward + residual

        # masked mean pool over real (non-padding) text tokens only
        mask = attention_mask.unsqueeze(-1).float()                              # (B, L, 1)
        txt_vec = (txt_ctx * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)   # (B, D)

        # the "bidirectional attention vector": both directions concatenated
        fused = torch.cat([img_vec, txt_vec], dim=1)                             # (B, 2*D)
        return self.classifier(fused)


def attn_fusion_forward(model, batch):
    images = batch["image"].to(device)
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    return model(images, input_ids, attention_mask)


attn_fusion_model = BiCrossAttentionFusion(NUM_CLASSES, CONFIG["text_model_name"])
history_attn = fit(attn_fusion_model, train_loader, val_loader, attn_fusion_forward,
                    CONFIG["epochs"], CONFIG["lr_fusion"], name="attention-fusion")
evaluate_and_record(attn_fusion_model, test_loader, attn_fusion_forward, "Attention fusion (bidirectional cross-attn)")


## Module 5 — Analysing the results

In [ ]:
results_df = pd.DataFrame(results).T.sort_values("test_acc", ascending=False)
results_df


In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(results_df.index, results_df["test_acc"])
plt.ylabel("Test accuracy")
plt.title("Image-only vs. text-only vs. fusion strategies — UPMC Food-101")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
for name, history in [
    ("image-only", history_img),
    ("text-only", history_txt),
    ("early-fusion", history_early),
    ("attention-fusion", history_attn),
]:
    plt.plot(history["val_acc"], marker="o", label=name)
plt.xlabel("epoch")
plt.ylabel("validation accuracy")
plt.title("Validation accuracy per epoch")
plt.legend()
plt.show()


##Module 5.1 Confusion matrix

Accuracy gives us one number. But one number doesn't tell us where the model succeeds or
fails, and it can't tell you whether a "1% improvement" is the model getting slightly better
everywhere, or dramatically fixing one specific kind of mistake while leaving everything else
unchanged.

So let's look at the confusion matrix instead: which true classes get mistaken for which
predicted classes, and how often. With 101 classes the full matrix is too big to read at once,
so we zoom into the handful of class *pairs* the image-only model confuses most — the ones most
likely to be visually similar dishes — and put the same pairs side by side for the multimodal
model.

If some of those confusions get smaller after adding text, that's a concrete, specific
explanation for the improvement — not just "the accuracy went up."


In [ ]:
# Predictions for the multimodal side of the comparison. Swap `attn_fusion_model` /
# `attn_fusion_forward` for `early_fusion_model` / `early_fusion_forward` to compare against
# early fusion instead -- the rest of this section works unchanged either way.
multi_preds_df = collect_predictions(attn_fusion_model, test_loader, test_final, attn_fusion_forward, id2label)
multi_model_name = "Attention fusion (bidirectional cross-attn)"

cm_img = confusion_matrix(image_preds_df["true_id"], image_preds_df["pred_id"], labels=range(NUM_CLASSES))
cm_multi = confusion_matrix(multi_preds_df["true_id"], multi_preds_df["pred_id"], labels=range(NUM_CLASSES))

# Find the most common (true -> predicted) mix-ups the IMAGE-ONLY model makes -- these are our
# candidates for "visually similar dishes." We rank by the image-only model on purpose: we want
# to compare the multimodal model against a fixed, image-defined notion of what's confusing.
offdiag = cm_img.copy()
np.fill_diagonal(offdiag, 0)
pair_counts = [
    (int(offdiag[i, j]), i, j)
    for i in range(NUM_CLASSES) for j in range(NUM_CLASSES)
    if i != j and offdiag[i, j] > 0
]
pair_counts.sort(reverse=True)
TOP_N_PAIRS = 8
top_pairs = pair_counts[:TOP_N_PAIRS]

print(f"Found {len(pair_counts)} distinct (true, predicted) confusion pairs from the image-only "
      f"model; zooming into the top {len(top_pairs)}.")
if len(top_pairs) == 0:
    print("No confusions to zoom into on this small test subsample -- try a larger "
          "samples_per_class_test so the image-only model actually makes a few mistakes.")


In [ ]:
if len(top_pairs) > 0:
    # The set of classes involved in those top confusions -- this is what we'll zoom the
    # confusion matrix into, instead of trying to look at all 101 classes at once.
    classes_of_interest = sorted({i for _, i, j in top_pairs} | {j for _, i, j in top_pairs})
    class_names_sub = [id2label[c] for c in classes_of_interest]

    cm_img_sub = cm_img[np.ix_(classes_of_interest, classes_of_interest)]
    cm_multi_sub = cm_multi[np.ix_(classes_of_interest, classes_of_interest)]

    def plot_confusion_subset(ax, cm_sub, class_names, title, vmax):
        im = ax.imshow(cm_sub, cmap="Blues", vmin=0, vmax=max(vmax, 1))
        ax.set_xticks(range(len(class_names)))
        ax.set_yticks(range(len(class_names)))
        ax.set_xticklabels(class_names, rotation=90, fontsize=8)
        ax.set_yticklabels(class_names, fontsize=8)
        ax.set_xlabel("predicted")
        ax.set_ylabel("true")
        ax.set_title(title, fontsize=11)
        for i in range(cm_sub.shape[0]):
            for j in range(cm_sub.shape[1]):
                val = cm_sub[i, j]
                if val > 0:
                    color = "white" if val > vmax * 0.6 else "black"
                    ax.text(j, i, str(int(val)), ha="center", va="center", fontsize=7, color=color)
        return im

    vmax = max(cm_img_sub.max(), cm_multi_sub.max())
    fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
    im = plot_confusion_subset(axes[0], cm_img_sub, class_names_sub, "Image-only: most-confused classes", vmax)
    plot_confusion_subset(axes[1], cm_multi_sub, class_names_sub, f"{multi_model_name}: same classes", vmax)
    fig.colorbar(im, ax=axes, fraction=0.025, pad=0.03, label="number of test examples")
    plt.suptitle("Confusion matrix, zoomed into the classes the image-only model confuses most", y=1.02)
    plt.show()


In [ ]:
# if len(top_pairs) > 0:
#     # The direct before/after answer to "did adding text fix THIS specific confusion?"
#     pair_rows = []
#     for count, i, j in top_pairs:
#         before = int(cm_img[i, j])
#         after = int(cm_multi[i, j])
#         pair_rows.append({
#             "true class": id2label[i],
#             "confused as": id2label[j],
#             "image-only count": before,
#             "multimodal count": after,
#             "change": after - before,
#         })
#     pair_df = pd.DataFrame(pair_rows)
#     display(pair_df)

#     decreased = int((pair_df["change"] < 0).sum())
#     same = int((pair_df["change"] == 0).sum())
#     increased = int((pair_df["change"] > 0).sum())
#     # print(f"Of the {len(pair_df)} most common image-only confusions: {decreased} decreased after "
#     #       f"adding text, {same} stayed the same, {increased} increased.")

#     if decreased > 0:
#         best = pair_df.sort_values("change").iloc[0]
#         print(f"Biggest improvement: '{best['true class']}' being mistaken for "
#               f"'{best['confused as']}' dropped from {best['image-only count']} to "
#               f"{best['multimodal count']}.")


## Module 5.2— Fusion succeeding where both modalities failed alone

Examples where the **image model is wrong AND the text model is wrong, but
the fusion model is right**. Neither modality alone contained enough information to solve these,
fusion had to combine partial, individually-insufficient evidence from both sides into
something neither one could produce on its own. That's not "more data helping a little," that's
the model actually synthesizing two incomplete signals into a complete one.


In [ ]:
# Reuse the already-computed per-example predictions for all three models: image_preds_df and
# text_preds_df (from the complementary-modalities section after Part B), and multi_preds_df
# (the fusion model's predictions, from Part 5b above). All three are row-aligned to `test_final`.
rescue_df = image_preds_df[["full_path", "text", "true_label"]].copy()
rescue_df["img_pred"] = image_preds_df["pred_label"].values
rescue_df["img_pred_prob"] = image_preds_df["pred_prob"].values
rescue_df["img_correct"] = image_preds_df["correct"].values
rescue_df["img_probs"] = image_preds_df["probs"].values

rescue_df["txt_pred"] = text_preds_df["pred_label"].values
rescue_df["txt_pred_prob"] = text_preds_df["pred_prob"].values
rescue_df["txt_correct"] = text_preds_df["correct"].values
rescue_df["txt_probs"] = text_preds_df["probs"].values

rescue_df["multi_pred"] = multi_preds_df["pred_label"].values
rescue_df["multi_pred_prob"] = multi_preds_df["pred_prob"].values
rescue_df["multi_correct"] = multi_preds_df["correct"].values
rescue_df["multi_probs"] = multi_preds_df["probs"].values

both_wrong_individually = rescue_df[~rescue_df["img_correct"] & ~rescue_df["txt_correct"]]
rescue_pool = both_wrong_individually[both_wrong_individually["multi_correct"]]

n_both_wrong = len(both_wrong_individually)
n_rescued = len(rescue_pool)
print(f"Examples where BOTH single-modality models were wrong: {n_both_wrong}")
if n_both_wrong > 0:
    print(f"Of those, {multi_model_name} got {n_rescued} correct "
          f"({n_rescued/n_both_wrong:.1%}) -- cases neither modality could solve alone, but the "
          f"combination could.")
else:
    print("No example in this small test subsample stumped both single-modality models at once, "
          "so there's nothing for fusion to 'rescue' here -- try a larger samples_per_class_test.")


In [ ]:
def show_rescue_gallery(examples_df, title, max_examples=3):
    """Like show_complementary_gallery, but with a fourth column: the fusion model's own
    prediction, so you can see all three models' top-3 classes on the same row."""
    n = min(len(examples_df), max_examples)
    if n == 0:
        print(f"{title}: no matching examples in this test subsample.")
        return

    fig, axes = plt.subplots(n, 4, figsize=(16, 4.0 * n))
    axes = np.atleast_2d(axes)

    for row, (_, example) in enumerate(examples_df.head(n).iterrows()):
        img = Image.open(example["full_path"]).convert("RGB")
        axes[row, 0].imshow(img)
        axes[row, 0].axis("off")
        caption = example["text"] if len(example["text"]) <= 40 else example["text"][:37] + "..."
        axes[row, 0].set_title(f"true: {example['true_label']}\ncaption: \"{caption}\"", fontsize=8)

        bar_panel(axes[row, 1], example["img_probs"], "image (wrong)", example["img_pred"], example["true_label"])
        bar_panel(axes[row, 2], example["txt_probs"], "text (wrong)", example["txt_pred"], example["true_label"])
        bar_panel(axes[row, 3], example["multi_probs"], "fusion (right)", example["multi_pred"], example["true_label"])

    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()


rescue_top = rescue_pool.sort_values("multi_pred_prob", ascending=False)
show_rescue_gallery(
    rescue_top,
    f"Rescued by fusion: image WRONG, text WRONG, {multi_model_name} CORRECT",
    max_examples=3,
)


## Exercise for you
You now have five numbers to compare: image-only, text-only, and three ways of fusing them.
On UPMC Food-101 the typical pattern is that every fusion strategy beats both single-modality
baselines, because the image and text branches tend to make different mistakes on different
examples. Which *fusion* strategy wins is less
consistent and depends on how much training data and how many epochs you give each model.

Try Yourself-

- Set `samples_per_class_train`/`samples_per_class_test` to `None` and train for more epochs on
  the full dataset (best done with a proper learning-rate schedule and early stopping).
- Swap the backbones for stronger ones (e.g. a bigger ResNet or ViT for images, a bigger
  transformer for text), or try CLIP's image/text encoders as a fourth, contrastively
  pretrained baseline.
- Look at which examples each modality gets wrong (a confusion matrix per model) to build
  intuition for why fusion helps on this dataset specifically.

